In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import json
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używam urządzenia: {device}")

PROJECT_DIR = '/content/drive/Shareddrives/adv_images/adversarial_images/'
LABELS_CLEAN_VGG = PROJECT_DIR + 'prediction_results/clean_VGG.json'

FEATURE_EXTRACTION_RESULTS = "/content/drive/MyDrive/TAI/sem2/DL/feature_extraction"

BATCH_SIZE = 1

Używam urządzenia: cuda


In [3]:
class AdversarialImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []

        if os.path.exists(root_dir):
            for file in os.listdir(root_dir):
                if file.endswith(".png"):
                    self.image_files.append(file)

        # Sortowanie po indeksie z nazwy (adv_image_0_label_3.png -> 0)
        try:
            self.image_files.sort(key=lambda x: int(x.split('_')[2]))
            self.image_file_names.sort(key=lambda x: int(x.split('_')[2]))
        except Exception as e:
            print("Błąd sortowania. Sprawdź nazwy plików.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        # Zwracamy sam obraz do modelu
        return image, self.image_files[idx]

In [5]:
class AttackMetadata:
    def __init__(self, attack_name: str, generated_by: str, model_attacked: str):
        self.attack_name = attack_name
        self.generated_by = generated_by
        self.model_attacked = model_attacked

    def to_dict(self) -> dict:
        return {"attack_name": self.attack_name, "generated_by": self.generated_by, "model_attacked": self.model_attacked}

In [7]:
class ImageInfo:
    def __init__(
        self,
        attack_metadata: AttackMetadata,
        image_index: int,
        true_label: int,
        clean_prediction_label: int,
        adversarial_prediction_label: int,
        clean_feature_map=None,
        adversarial_feature_map=None
    ):
        self.image_index = image_index
        self.true_label = true_label

        self.clean_prediction_label = clean_prediction_label
        self.adversarial_prediction_label = adversarial_prediction_label

        self.clean_feature_map = clean_feature_map
        self.adversarial_feature_map = adversarial_feature_map

        self.attack_metadata = attack_metadata

    def is_clean_prediction_correct(self) -> bool:
        return self.clean_prediction_label == self.true_label

    def is_adversarial_prediction_correct(self) -> bool:
        return self.adversarial_prediction_label == self.true_label

    def was_attack_successful(self) -> bool:
        return (self.clean_prediction_label == self.true_label and self.adversarial_prediction_label != self.true_label)

    def __repr__(self) -> str:
        return (
            f"ImageInfo("
            f"image_index={self.image_index}, "
            f"true_label={self.true_label}, "
            f"clean_prediction_label={self.clean_prediction_label}, "
            f"adversarial_prediction_label={self.adversarial_prediction_label}, "
            f"attack_name={self.attack_metadata.attack_name}, "
            f"clean_feature_map_set={self.clean_feature_map is not None}, "
            f"adversarial_feature_map_set={self.adversarial_feature_map is not None}"
            f")"
        )

    def to_dict(self) -> dict:
        data = {
            "image_index": self.image_index,
            "true_label": self.true_label,

            "clean_prediction_label": self.clean_prediction_label,
            "adversarial_prediction_label": self.adversarial_prediction_label,

            "is_clean_prediction_correct": self.is_clean_prediction_correct(),
            "is_adversarial_prediction_correct": self.is_adversarial_prediction_correct(),
            "was_attack_successful": self.was_attack_successful(),
            "attack_metadata": self.attack_metadata.to_dict()
        }
        # data.update(self.get_feature_map_shapes())
        return data


In [4]:
mean, std = [0.4914, 0.4822, 0.4465], [0.247, 0.2435, 0.2616]

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

clean_dataset = CIFAR10(root='./data', train=False, download=True, transform=preprocess)
clean_loader = DataLoader(clean_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Dane gotowe. Clean: {len(clean_dataset)}")

100%|██████████| 170M/170M [00:03<00:00, 44.2MB/s]


Błąd sortowania. Sprawdź nazwy plików.
Dane gotowe. Clean: 10000, Adv: 10000


In [13]:
# model for attack genration | attack name | model being attacked:

# Resnet | OnePixel | VGG
LABELS_RESNET_ONEPIXEL_VGG = PROJECT_DIR + 'prediction_results/ResNet_OnePixel_VGG.json'
IMAGES_RESNET_ONEPIXEL_VGG_DIR = PROJECT_DIR + 'test_ResNet_OnePixel'

# NAMING: adv_{model_attack_genration}_{attack_name}_{model_being_attacked}_{object_name}
adv_resnet_onepixel_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_ONEPIXEL_VGG_DIR, transform=preprocess)
adv_resnet_onepixel_vgg_loader = DataLoader(adv_resnet_onepixel_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | PGD | VGG
LABELS_RESNET_PGD_VGG = PROJECT_DIR + 'prediction_results/ResNet_PGD_VGG.json'
IMAGES_RESNET_PGD_VGG_DIR = PROJECT_DIR + 'test_ResNet_PGD'

adv_resnet_pgd_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_PGD_VGG_DIR, transform=preprocess)
adv_resnet_pgd_vgg_loader = DataLoader(adv_resnet_pgd_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | FGSM | VGG
LABELS_RESNET_FGSM_VGG = PROJECT_DIR + 'prediction_results/ResNet_FGSM_VGG.json'
IMAGES_RESNET_FGSM_VGG_DIR = PROJECT_DIR + 'test_ResNet_FGSM'

adv_resnet_fgsm_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_FGSM_VGG_DIR, transform=preprocess)
adv_resnet_fgsm_vgg_loader = DataLoader(adv_resnet_fgsm_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | DeepFool | VGG
LABELS_RESNET_DEEPFOOL_VGG = PROJECT_DIR + 'prediction_results/ResNet_DeepFool_VGG.json'
IMAGES_RESNET_DEEPFOOL_VGG_DIR = PROJECT_DIR + 'test_ResNet_DeepFool'

adv_resnet_deepfool_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_DEEPFOOL_VGG_DIR, transform=preprocess)
adv_resnet_deepfool_vgg_loader = DataLoader(adv_resnet_deepfool_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | CW | VGG
LABELS_RESNET_CW_VGG = PROJECT_DIR + 'prediction_results/ResNet_CW_VGG.json'
IMAGES_RESNET_CW_VGG_DIR = PROJECT_DIR + 'test_ResNet_CW'

adv_resnet_cw_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_CW_VGG_DIR, transform=preprocess)
adv_resnet_cw_vgg_loader = DataLoader(adv_resnet_cw_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | AutoAttack | VGG
LABELS_RESNET_AUTOATTACK_VGG = PROJECT_DIR + 'prediction_results/ResNet_AutoAttack_VGG.json'
IMAGES_RESNET_AUTOATTACK_VGG_DIR = PROJECT_DIR + 'test_ResNet_AutoAttack'

adv_resnet_autoattack_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_AUTOATTACK_VGG_DIR, transform=preprocess)
adv_resnet_autoattack_vgg_loader = DataLoader(adv_resnet_autoattack_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [8]:
with open(LABELS_CLEAN_VGG, 'r') as f:
    labels_clean_vgg = json.load(f)

with open(LABELS_RESNET_ONEPIXEL_VGG, 'r') as f:
    labels_resnet_onepixel_vgg = json.load(f)

labels_resnet_onepixel_vgg = labels_resnet_onepixel_vgg['ResNet_OnePixel_VGG']
labels_clean_vgg = labels_clean_vgg['clean_VGG']

metadata_resnet_onepixel_vgg = AttackMetadata("OnePixel", "ResNet", "VGG")

In [11]:
imageinfos_resnet_onepixel_vgg = []

for idx, filename in enumerate(adv_resnet_onepixel_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index = int(match.group(1))
      true_label = int(match.group(2))

      # attak_metadata, index, label, json clean label, json attack label, (featuremap clean), (featuremap attacked)
      imageinfos_resnet_onepixel_vgg.append(
          ImageInfo(metadata_resnet_onepixel_vgg,
                    image_index,
                    true_label,
                    labels_clean_vgg[idx],
                    labels_resnet_onepixel_vgg[idx]
          )
      )

print(f"Processed {len(imageinfos_resnet_onepixel_vgg)} images and created ImageInfo objects.")

Processed 10000 images and created ImageInfo objects.


In [12]:
num_examples_to_print = 100
print(f"\nFirst {num_examples_to_print} image prediction infos (calling .to_dict()):")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_onepixel_vgg[i].to_dict())


First 100 image prediction infos (calling .to_dict()):
{'image_index': 0, 'true_label': 3, 'clean_prediction_label': 3, 'adversarial_prediction_label': 3, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': True, 'was_attack_successful': False, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_index': 1, 'true_label': 8, 'clean_prediction_label': 8, 'adversarial_prediction_label': 5, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': False, 'was_attack_successful': True, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_index': 2, 'true_label': 8, 'clean_prediction_label': 8, 'adversarial_prediction_label': 8, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': True, 'was_attack_successful': False, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_ind

# TODO: zamienić gotowy mobilenet na wytrenowany inception !!!

In [ ]:
# MobileNetV2_x1_0 inna architektura niż VGG/ResNet (Depthwise Separable Convs)
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_mobilenetv2_x1_0", pretrained=True)

# W tym modelu ostatnia warstwa nazywa się 'classifier'.
# Zamieniamy ją na Identity, aby uzyskać wektor cech.
# Dla MobileNetV2 na CIFAR-10 wektor cech ma zwykle 1280 wymiarów.
model.classifier = nn.Identity()
model.to(device)
model.eval()
print("Model MobileNetV2 gotowy. Ostatnia warstwa usunięta.")

### TODO: Uzupełenienie feature map w batch > 1

In [ ]:
@torch.no_grad()
def extract_feature_map(model, image_tensor, device):
    model.eval()
    image_tensor = image_tensor.to(device)
    fmap = model(image_tensor)              # np. (1, 1280, 7, 7)
    return fmap.cpu().squeeze(0)           # (1280, 7, 7)

In [ ]:
for idx, (img, filename) in enumerate(clean_loader):
    fmap = extract_feature_map(model, img, device)  # (C, H, W)
    imageinfos_resnet_onepixel_vgg[idx].clean_feature_map = fmap

In [ ]:
for idx, (img, filename) in enumerate(adv_resnet_onepixel_vgg_loader):
    fmap = extract_feature_map(model, img, device)
    imageinfos_resnet_onepixel_vgg[idx].one_pixel_adv_feature_map = fmap
